# READ FILE

In [0]:
path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Reseller.csv"

df = (spark.read
      .option("header", True)
      .option("sep", "\t")
      .option("mode", "PERMISSIVE")
      .csv(path))

print("Loaded reseller file:", path)
print("Columns:", df.columns)
df.printSchema()
display(df)

### Problems:
    
    1.Change Column Names to snake_case

## Appropriate data typing check

In [0]:
from pyspark.sql import functions as F

typed_check = df.select(
    "ResellerKey",
    F.when(F.col("ResellerKey").rlike(r"^\d+$"), F.col("ResellerKey").cast("int")).otherwise(F.lit(None)).alias("ResellerKey_as_int")
)

invalid_key = typed_check.filter(F.col("ResellerKey_as_int").isNull())
print("Typing check: ResellerKey should be an integer")
print("Rows with invalid/non-integer ResellerKey:", invalid_key.count())
display(invalid_key.select("ResellerKey").distinct().limit(100))

### Problems
1.ResellerKey should be an integer

## Missing Values

In [0]:
from pyspark.sql import functions as F

critical = ["ResellerKey", "Business Type", "Reseller", "City", "State-Province", "Country-Region"]

missing_counts = df.select([
    F.sum((F.col(c).isNull() | (F.length(F.trim(F.col(c))) == 0)).cast("int")).alias(c)
    for c in critical
])

print("Missing/blank check: counts for critical fields")
display(missing_counts)

## Duplicate Keys

In [0]:
from pyspark.sql import functions as F

dup_keys = (df.groupBy("ResellerKey")
              .count()
              .filter(F.col("count") > 1)
              .orderBy(F.col("count").desc()))

print("Join key uniqueness check: duplicate ResellerKey values:", dup_keys.count())
display(dup_keys.limit(100))

## Text standardization need: leading/trailing whitespace

In [0]:
from pyspark.sql import functions as F

text_cols = ["Business Type", "Reseller", "City", "State-Province", "Country-Region"]

ws_issues = df.select([
    F.sum((F.col(c).isNotNull() & (F.col(c) != F.trim(F.col(c)))).cast("int")).alias(c)
    for c in text_cols
])

print("Standardization check: leading/trailing whitespace issues per column")
display(ws_issues)

## Inconsistent values: Country naming variations

In [0]:
from pyspark.sql import functions as F

print("Country naming consistency check")

display(
    df.groupBy("Country-Region")
      .count()
      .orderBy(F.col("count").desc())
)

### Problems
UK must be United Kindom